# gm/ID lookup-table tour — the committed tri-PDK LUTs

The analog-db ships **pre-computed gm/ID lookup tables** (Phase 6a): each PDK's core nmos is
characterized over an `(L × VGS × VDS × VSB)` grid with an automated ngspice testbench and stored as
a **pygmid-compatible `.pkl`** at `_shared/gmid/<pdk>/<device>__<corner>.pkl`. This notebook is the
LUT layer's test/experimentation surface: load the tables, inspect them, and reproduce the canonical
gm/ID design curves — including a **cross-PDK comparison** you can't get from any single-technology kit.

- Generation: `analog-db gmid-extract --pdk <pdk>` (docs: `_shared/GMID.md`) — corner, LV/HV device
  variant, and the full grid are configurable.
- Reader: [`pygmid`](https://github.com/dreoilin/pygmid) (`Lookup`), the Python port of the
  book's MATLAB kit ([Jespers & Murmann](https://github.com/bmurmann/Book-on-gm-ID-design)).
- Worked sizing patterns: [iic-jku/analog-circuit-design](https://github.com/iic-jku/analog-circuit-design);
  the sizing flow itself is the companion notebook `gmid_sizing_demo.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pygmid import Lookup

from spicexplorer_analog_db import paths, pdks

%matplotlib inline
plt.rcParams["figure.figsize"] = (11, 3.6)

GMID_ROOT = paths.shared_root() / "gmid"
luts = {p.parent.name: Lookup(str(p)) for p in sorted(GMID_ROOT.glob("*/*.pkl"))}
print("committed LUTs:")
for pdk, lk in luts.items():
    print(f"  {pdk:12} {lk['INFO']}")

## What's inside a table

A flat dict: four **axis vectors** (`L` in µm, `VGS`/`VDS`/`VSB` in V) and 4-D arrays indexed
`[L, VGS, VDS, VSB]` — DC operating point (`ID VT GM GMB GDS`), capacitances (`CGG CGS CGD CGB CDD
CSS`), and noise PSDs (`STH SFL`) — plus the characterization header (`CORNER TEMP W NFING`).
Everything below derives from these arrays.

In [ ]:
rows = []
for pdk, lk in luts.items():
    rows.append({
        "pdk": pdk, "corner": lk["CORNER"], "T (K)": lk["TEMP"], "W (µm)": lk["W"],
        "L grid (µm)": f"{lk['L'][0]:g}…{lk['L'][-1]:g} ({len(lk['L'])} pts)",
        "VGS": f"0…{lk['VGS'][-1]:g} ({len(lk['VGS'])} pts)",
        "VDS": f"0…{lk['VDS'][-1]:g} ({len(lk['VDS'])} pts)",
        "VSB pts": len(lk["VSB"]),
    })
pd.DataFrame(rows).set_index("pdk")

## The canonical design curves

**gm/ID vs VGS** — the transconductance-efficiency curve. Weak inversion plateaus near
~25–30 S/A, strong inversion falls toward a few S/A; this is the knob the whole methodology
turns. (Curves at mid-rail VDS, VSB = 0.)

In [ ]:
fig, axes = plt.subplots(1, len(luts), sharey=True)
for ax, (pdk, lk) in zip(np.atleast_1d(axes), luts.items()):
    vds_i = len(lk["VDS"]) // 2
    for li in [0, len(lk["L"]) // 2, len(lk["L"]) - 1]:
        gm, idd = lk["GM"][li, :, vds_i, 0], lk["ID"][li, :, vds_i, 0]
        with np.errstate(divide="ignore", invalid="ignore"):
            eff = np.where(idd > 0, gm / idd, np.nan)
        ax.plot(lk["VGS"], eff, label=f"L={lk['L'][li]:g}µm")
    ax.set_title(pdk); ax.set_xlabel("VGS (V)"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("gm/ID (S/A)")
plt.tight_layout()

**gm/ID vs current density JD = ID/W** — the master sizing chart: pick gm/ID, read JD, and width
follows from `W = ID / JD`. Longer channels shift the curve left (less current per µm at the same
efficiency).

In [ ]:
fig, axes = plt.subplots(1, len(luts), sharey=True)
for ax, (pdk, lk) in zip(np.atleast_1d(axes), luts.items()):
    vds_i = len(lk["VDS"]) // 2
    for li in [0, len(lk["L"]) // 2, len(lk["L"]) - 1]:
        gm, idd = lk["GM"][li, :, vds_i, 0], lk["ID"][li, :, vds_i, 0]
        with np.errstate(divide="ignore", invalid="ignore"):
            eff = np.where(idd > 0, gm / idd, np.nan)
        ax.semilogx(idd / lk["W"], eff, label=f"L={lk['L'][li]:g}µm")
    ax.set_title(pdk); ax.set_xlabel("JD = ID/W (A/µm)"); ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("gm/ID (S/A)")
plt.tight_layout()

**The two tradeoff curves** — transit frequency `fT = gm/(2π·CGG)` (speed) and intrinsic gain
`gm/gds` (accuracy) against gm/ID. Together they frame every sizing compromise: low gm/ID buys
speed, high gm/ID buys efficiency/swing, long L buys gain at the cost of fT.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2)
for pdk, lk in luts.items():
    vds_i = len(lk["VDS"]) // 2
    for li, ls in [(0, "-"), (len(lk["L"]) - 1, "--")]:
        gm, idd = lk["GM"][li, :, vds_i, 0], lk["ID"][li, :, vds_i, 0]
        cgg, gds = lk["CGG"][li, :, vds_i, 0], lk["GDS"][li, :, vds_i, 0]
        with np.errstate(divide="ignore", invalid="ignore"):
            eff = np.where(idd > 0, gm / idd, np.nan)
            ft = np.where(cgg > 0, gm / (2 * np.pi * cgg), np.nan)
            av = np.where(gds > 0, gm / gds, np.nan)
        lbl = f"{pdk} L={lk['L'][li]:g}"
        ax1.semilogy(eff, ft, ls, label=lbl, alpha=0.8)
        ax2.plot(eff, av, ls, label=lbl, alpha=0.8)
ax1.set_xlabel("gm/ID (S/A)"); ax1.set_ylabel("fT (Hz)"); ax1.grid(alpha=0.3, which="both")
ax2.set_xlabel("gm/ID (S/A)"); ax2.set_ylabel("intrinsic gain gm/gds"); ax2.grid(alpha=0.3)
ax1.legend(fontsize=7); ax2.legend(fontsize=7)
plt.tight_layout()

## Cross-PDK comparison at matched L

All three L-grids contain **0.5 µm** — a direct apples-to-apples overlay of the three
technologies' efficiency-vs-density tradeoff (something the single-PDK book kits can't show).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for pdk, lk in luts.items():
    li = int(np.argmin(np.abs(lk["L"] - 0.5)))
    vds_i = len(lk["VDS"]) // 2
    gm, idd = lk["GM"][li, :, vds_i, 0], lk["ID"][li, :, vds_i, 0]
    with np.errstate(divide="ignore", invalid="ignore"):
        eff = np.where(idd > 0, gm / idd, np.nan)
    ax.semilogx(idd / lk["W"], eff, label=f"{pdk} (L={lk['L'][li]:g}µm, VDS={lk['VDS'][vds_i]:g}V)")
ax.set_xlabel("JD = ID/W (A/µm)"); ax.set_ylabel("gm/ID (S/A)")
ax.set_title("Three PDKs, one chart — core nmos @ tt, L = 0.5 µm")
ax.grid(alpha=0.3, which="both"); ax.legend()
plt.tight_layout()

## pygmid lookups (the sizing API the tables serve)

`Lookup` interpolates the grid: raw quantities vs a bias (`GM` at a VGS), **ratio-vs-ratio** mode
(`ID_W` at a target `GM_ID` — the sizing workhorse), and the inverse `look_upVGS`. Note: pass
**scalar L** (loop for L sweeps — vectorization is over GM_ID/VGS).

In [ ]:
nch = luts["sky130"]
gm_id = np.array([5, 10, 15, 20, 25])
jd = nch.look_up("ID_W", GM_ID=gm_id, VDS=0.9, VSB=0, L=0.5)     # A/µm at each efficiency
vgs10 = float(nch.look_upVGS(GM_ID=10, VDS=0.9, VSB=0, L=0.5))   # bias for gm/ID = 10
ft10 = float(nch.look_up("GM_CGG", GM_ID=10, VDS=0.9, VSB=0, L=0.5)) / (2 * np.pi)
pd.DataFrame({"gm/ID (S/A)": gm_id, "JD (A/µm)": np.asarray(jd)}).set_index("gm/ID (S/A)").T \
    .style.format("{:.3e}")

In [ ]:
print(f"at gm/ID=10, L=0.5µm, VDS=0.9V:  VGS = {vgs10:.3f} V,  fT = {ft10/1e9:.2f} GHz")

## PDK passives (the other half of sizing)

Sizing also needs to turn a target R into squares and a target C into MIM area. The measured tt
constants live in each PDK registry (`_shared/pdk/<pdk>.yaml` → `passives.models`).

In [ ]:
rows = []
for pdk in luts:
    reg = pdks.load_registry(pdk)
    for model, info in (reg.get("passives", {}).get("models", {}) or {}).items():
        rows.append({"pdk": pdk, "model": model, "kind": info["kind"],
                     "sheet_res (Ω/□)": info.get("sheet_res", ""),
                     "area_cap (F/µm²)": info.get("area_cap", "")})
pd.DataFrame(rows).set_index(["pdk", "model"])

## Regenerating / extending the tables

Needs the EDA base image (`docker compose --profile base build spice-base` in spicexplorer-platform):

```bash
analog-db gmid-extract --pdk sky130                                  # re-write the committed LUT
analog-db gmid-extract --pdk sky130 --device sky130_fd_pr__pfet_01v8 # add the pmos
analog-db gmid-extract --pdk ihp-sg13g2 --device sg13_hv_nmos        # HV variant (corner lib auto-swaps)
analog-db gmid-extract --pdk gf180mcu --corner ss --vgs 0,0.025,3.3  # corner + finer grid
```

Every knob (device incl. LV/HV variant, corner, grids, W, fingers, temperature) is configurable —
see `_shared/GMID.md`. **What's next:** `gmid_sizing_demo.ipynb` applies these tables to actual
device sizing; the typed `spicexplorer-gmid` platform tool grows out of these patterns
(meta-repo `doc/plan_gmid_sizing.md`).